# YieldSAT Crop-Mapping Feature Store

This notebook extracts a broad, reusable feature store from the YieldSAT raw dataset for crop-mapping research. It uses Dask to process field chunks in parallel, then writes only two final files: one field-level table and one Sentinel-2 observation-level table with weather joined into each observation.


## 1. Install and Import Dependencies

This cell checks for the packages needed to read GeoTIFF, CSV, JSON, Parquet files, and run Dask locally on Kaggle. Kaggle usually has most of them installed, but the check keeps the notebook repeatable.


In [ ]:
import importlib.util
import subprocess
import sys

missing = []
for package_name, import_name in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("rasterio", "rasterio"),
    ("tqdm", "tqdm"),
    ("pyarrow", "pyarrow"),
]:
    if importlib.util.find_spec(import_name) is None:
        missing.append(package_name)

if importlib.util.find_spec("dask") is None or importlib.util.find_spec("distributed") is None:
    missing.append("dask[distributed]")

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")


## 2. Configure the Full Extraction Run

Set the dataset root and output directory before running the extractor. On Kaggle, attach `abdulhamedeid/yieldsat-private` and leave `DATA_ROOT` as `None` to auto-detect the input path. This notebook always processes the full attached dataset.


In [ ]:
from pathlib import Path

# Set this only if Kaggle auto-detection fails. Otherwise keep None.
DATA_ROOT = None

OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("data/processed")
OUTPUT_DIR = OUTPUT_ROOT / "yieldsat_crop_mapping_feature_store"
TEMP_PARTS_DIR = OUTPUT_DIR / "_parts"

# Kaggle run controls. The attached runtime has 4 CPU cores, so use 4 single-thread workers.
DASK_N_WORKERS = 4
DASK_THREADS_PER_WORKER = 1
CHUNK_SIZE = 50

# SCL classes kept as usable observations. These follow the earlier YieldSAT normalization work.
SCL_KEEP_CLASSES = (2, 4, 5, 6)  # dark area, vegetation, bare soil, water

# Keep False for only the two final Parquet files. Set True if you want a download zip too.
SAVE_ZIP = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)
print("Dask workers:", DASK_N_WORKERS)
print("Chunk size:", CHUNK_SIZE)


## 3. Define Core Metadata Helpers

These helpers find YieldSAT field folders without scanning every raster file, parse inconsistent metadata schemas, and create season-position fields used by both weather and Sentinel-2 observations.

In [ ]:
import json
import math
import re
import time
import traceback
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm

S2_PATTERN = re.compile(r"S2_L2A_(\d{4})(\d{2})(\d{2})\.tif$")
SCL_PATTERN = re.compile(r"S2_L2A_SCL_(\d{4})(\d{2})(\d{2})\.tif$")

BAND_ORDER = [
    "B01", "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B09", "B11", "B12",
]

SCL_CLASSES = {
    0: "nodata",
    1: "saturated_or_defective",
    2: "dark_area",
    3: "cloud_shadow",
    4: "vegetation",
    5: "bare_soil",
    6: "water",
    7: "cloud_low_probability",
    8: "cloud_medium_probability",
    9: "cloud_high_probability",
    10: "thin_cirrus",
    11: "snow_or_ice",
}

@dataclass
class CropFeatureStoreConfig:
    scl_keep_classes: tuple[int, ...] = SCL_KEEP_CLASSES


def resolve_data_root(data_root: Optional[str | Path]) -> Path:
    if data_root is not None:
        root = Path(data_root)
        if not root.exists():
            raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
        return root

    candidates = [
        Path("/kaggle/input/yieldsat-private"),
        Path("/kaggle/input/abdulhamedeid/yieldsat-private"),
        Path("/kaggle/input/datasets/abdulhamedeid/yieldsat-private"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError("Could not auto-detect YieldSAT dataset. Set DATA_ROOT explicitly.")


def discover_fields(data_root: Path) -> list[Path]:
    raw_root = data_root / "Raw" if (data_root / "Raw").exists() else data_root
    patterns = [
        "*/*/*/metadata-*.json",  # Raw/Country/Country/field/metadata
        "*/*/metadata-*.json",    # Raw/Country/field/metadata
    ]
    paths = []
    seen = set()
    for pattern in patterns:
        for path in sorted(raw_root.glob(pattern)):
            if path not in seen:
                paths.append(path)
                seen.add(path)
    return paths


def parse_field_shared_name(shared_name: str) -> dict:
    parts = shared_name.split("_")
    if len(parts) >= 6:
        return {
            "country": parts[0],
            "provider": parts[1],
            "farm": parts[2],
            "field_num": parts[3],
            "crop": parts[4].lower(),
            "year": int(parts[5]) if parts[5].isdigit() else None,
        }
    return {}


def parse_date(value) -> Optional[date]:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    for fmt in ("%d.%m.%Y", "%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            continue
    return None


def season_position(obs_date: date, seeding: Optional[date], harvest: Optional[date]) -> dict:
    if obs_date is None or seeding is None or harvest is None:
        return {
            "days_after_seeding": np.nan,
            "days_before_harvest": np.nan,
            "season_progress": np.nan,
            "in_declared_season": False,
        }
    season_days = (harvest - seeding).days
    days_after = (obs_date - seeding).days
    days_before = (harvest - obs_date).days
    progress = days_after / season_days if season_days > 0 else np.nan
    return {
        "days_after_seeding": days_after,
        "days_before_harvest": days_before,
        "season_progress": progress,
        "in_declared_season": bool(seeding <= obs_date <= harvest),
    }


def parse_metadata(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    shared_name = raw.get("field_shared_name", "") or path.stem.replace("metadata-", "")
    identity = parse_field_shared_name(shared_name)
    adm = raw.get("adm_units", {}) or {}
    geo = raw.get("geometry", {}) or {}
    seeding = parse_date(raw.get("seeding_date"))
    harvest = parse_date(raw.get("harvesting_date"))

    return {
        "field_id": path.stem.replace("metadata-", ""),
        "field_shared_name": shared_name,
        "country": identity.get("country") or raw.get("country") or adm.get("country", ""),
        "provider": identity.get("provider") or raw.get("data_provider", ""),
        "farm": identity.get("farm", ""),
        "field_num": identity.get("field_num", ""),
        "crop": (identity.get("crop") or raw.get("crop", "")).lower(),
        "year": identity.get("year") or raw.get("year"),
        "adm_country": adm.get("country", ""),
        "adm_1": adm.get("adm_1") or adm.get("adm_unit0", ""),
        "adm_2": adm.get("adm_2") or adm.get("adm_unit1", ""),
        "adm_3": adm.get("adm_unit2", ""),
        "adm_4": adm.get("adm_unit3", ""),
        "seeding_date": seeding.isoformat() if seeding else None,
        "harvesting_date": harvest.isoformat() if harvest else None,
        "season_length_days": (harvest - seeding).days if seeding and harvest else np.nan,
        "area_ground_truth_ha": raw.get("area_ground_truth"),
        "area_calculated_ha": raw.get("area_calculated"),
        "centroid_lat": raw.get("centroid_latitude_wgs84") or geo.get("lat"),
        "centroid_lon": raw.get("centroid_longitude_wgs84") or geo.get("lon"),
        "crs_epsg": raw.get("projected_crs_epsg"),
        "moisture_pct": raw.get("standard_moisture"),
        "yieldmap_quality": raw.get("yieldmap_quality"),
        "quality_density": raw.get("quality_density"),
        "seeding_date_type": raw.get("seeding_date_type", ""),
        "metadata_path": str(path),
    }


def metadata_dates(meta: dict) -> tuple[Optional[date], Optional[date]]:
    return parse_date(meta.get("seeding_date")), parse_date(meta.get("harvesting_date"))


def is_missing_scalar(value) -> bool:
    if value is None:
        return True
    try:
        result = pd.isna(value)
    except (TypeError, ValueError):
        return False
    if isinstance(result, (bool, np.bool_)):
        return bool(result)
    return False


def prepare_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            non_null = df[col].dropna()
            if not non_null.empty and non_null.map(lambda value: isinstance(value, (bool, np.bool_))).all():
                df[col] = df[col].astype("boolean")
            else:
                df[col] = df[col].map(lambda value: None if is_missing_scalar(value) else str(value))
    return df


## 4. Extract Static Raster and Weather Join Features

Static raster features summarize DEM and soil layers once per field. Weather is loaded only to join daily, rolling, and cumulative weather features onto each Sentinel-2 observation.


In [ ]:
from datetime import timedelta

RASTER_AGGS = ("mean", "std", "min", "max", "p10", "p25", "p75", "p90", "count")
DEM_LAYERS = ("dem", "aspect", "slope", "curvature", "twi")
SOIL_LAYERS = (
    "cec_0_200cm", "cfvo_0_200cm", "clay_0_200cm", "nitrogen_0_200cm",
    "phh2o_0_200cm", "sand_0_200cm", "silt_0_200cm", "soc_0_200cm",
)

WEATHER_JOIN_COLUMNS = [
    "weather_daily_Temp_mean_C",
    "weather_daily_Temp_max_C",
    "weather_daily_Temp_min_C",
    "weather_daily_Total_prec_mm",
    "weather_daily_gdd_base10",
    "weather_daily_rainy_day",
    "weather_daily_heavy_rain_day",
    "weather_daily_heat_stress_day",
    "weather_daily_cold_stress_day",
    "weather_cum_Total_prec_mm",
    "weather_cum_gdd_base10",
    "weather_cum_rainy_days",
    "weather_cum_heavy_rain_days",
    "weather_cum_heat_stress_days",
    "weather_cum_cold_stress_days",
]
for window_days in (7, 14, 30):
    WEATHER_JOIN_COLUMNS.extend([
        f"weather_{window_days}d_Total_prec_mm",
        f"weather_{window_days}d_Temp_mean_C",
        f"weather_{window_days}d_gdd_base10",
        f"weather_{window_days}d_rainy_days",
    ])


def array_stats(values: np.ndarray, prefix: str = "") -> dict:
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {f"{prefix}{agg}": np.nan for agg in RASTER_AGGS}
    return {
        f"{prefix}mean": float(np.mean(values)),
        f"{prefix}std": float(np.std(values)),
        f"{prefix}min": float(np.min(values)),
        f"{prefix}max": float(np.max(values)),
        f"{prefix}p10": float(np.percentile(values, 10)),
        f"{prefix}p25": float(np.percentile(values, 25)),
        f"{prefix}p75": float(np.percentile(values, 75)),
        f"{prefix}p90": float(np.percentile(values, 90)),
        f"{prefix}count": int(values.size),
    }


def normalize_raster_values(values: np.ndarray, layer_name: str) -> np.ndarray:
    if values.size == 0:
        return values
    layer = layer_name.lower()
    median_abs = float(np.nanmedian(np.abs(values)))
    if layer == "slope" and median_abs > 100:
        return values / 10000.0
    if layer == "aspect" and median_abs > 360:
        return values / 10000.0
    return values


def raster_stats(path: Path, layer_name: str) -> dict:
    try:
        with rasterio.open(path) as src:
            output = {}
            for band_idx in range(1, src.count + 1):
                arr = src.read(band_idx).astype("float64")
                mask = np.isfinite(arr)
                if src.nodata is not None:
                    mask &= arr != src.nodata
                values = normalize_raster_values(arr[mask], layer_name)
                prefix = "" if src.count == 1 else f"band{band_idx:02d}_"
                output.update(array_stats(values, prefix=prefix))
            return output
    except Exception:
        return {agg: np.nan for agg in RASTER_AGGS}


def find_layer_file(directory: Path, layer: str, field_name: str) -> Optional[Path]:
    direct = directory / f"{layer}-{field_name}.tif"
    if direct.exists():
        return direct
    matches = sorted(directory.glob(f"{layer}*.tif"))
    return matches[0] if matches else None


def extract_static_features(field_dir: Path, field_id: str) -> dict:
    features = {"field_id": field_id}

    dem_dir = field_dir / "dem"
    if dem_dir.exists():
        for layer in DEM_LAYERS:
            tif = find_layer_file(dem_dir, layer, field_dir.name)
            if tif:
                for key, value in raster_stats(tif, layer).items():
                    features[f"dem_{layer}_{key}"] = value

    soil_dir = field_dir / "soil"
    if soil_dir.exists():
        for layer in SOIL_LAYERS:
            tif = find_layer_file(soil_dir, layer, field_dir.name)
            if tif:
                for key, value in raster_stats(tif, layer).items():
                    features[f"soil_{layer}_{key}"] = value

    return features


def normalize_weather_units(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for source, dest in [
        ("Temp_mean", "Temp_mean_C"),
        ("Temp_max", "Temp_max_C"),
        ("Temp_min", "Temp_min_C"),
    ]:
        if source in df.columns:
            values = pd.to_numeric(df[source], errors="coerce")
            if values.dropna().median() > 100:
                values = values - 273.15
            df[dest] = values

    if "Total_prec" in df.columns:
        values = pd.to_numeric(df["Total_prec"], errors="coerce")
        if values.dropna().max() <= 1.0:
            values = values * 1000.0
        df["Total_prec_mm"] = values

    return df


def load_weather_daily(field_dir: Path, meta: dict) -> pd.DataFrame:
    weather_dir = field_dir / "weather"
    if not weather_dir.exists():
        return pd.DataFrame()

    csv_path = weather_dir / f"{meta['field_id']}.csv"
    if not csv_path.exists():
        matches = sorted(weather_dir.glob("*.csv"))
        if not matches:
            return pd.DataFrame()
        csv_path = matches[0]

    try:
        df = pd.read_csv(csv_path, parse_dates=["Date"])
    except Exception:
        return pd.DataFrame()
    if "Date" not in df.columns:
        return pd.DataFrame()

    df = normalize_weather_units(df).sort_values("Date").copy()
    df["obs_date"] = df["Date"].dt.date
    df["gdd_base10"] = (df.get("Temp_mean_C", np.nan) - 10.0).clip(lower=0)
    df["rainy_day"] = df.get("Total_prec_mm", np.nan) >= 1.0
    df["heavy_rain_day"] = df.get("Total_prec_mm", np.nan) >= 20.0
    df["heat_stress_day"] = df.get("Temp_max_C", np.nan) >= 35.0
    df["cold_stress_day"] = df.get("Temp_min_C", np.nan) <= 5.0
    return df


def weather_features_for_date(weather_df: pd.DataFrame, obs_date: date, seeding: Optional[date]) -> dict:
    features = {name: np.nan for name in WEATHER_JOIN_COLUMNS}
    if weather_df.empty or obs_date is None:
        return features

    daily = weather_df[weather_df["obs_date"] == obs_date]
    if not daily.empty:
        row = daily.iloc[0]
        features.update({
            "weather_daily_Temp_mean_C": row.get("Temp_mean_C", np.nan),
            "weather_daily_Temp_max_C": row.get("Temp_max_C", np.nan),
            "weather_daily_Temp_min_C": row.get("Temp_min_C", np.nan),
            "weather_daily_Total_prec_mm": row.get("Total_prec_mm", np.nan),
            "weather_daily_gdd_base10": row.get("gdd_base10", np.nan),
            "weather_daily_rainy_day": bool(row.get("rainy_day", False)),
            "weather_daily_heavy_rain_day": bool(row.get("heavy_rain_day", False)),
            "weather_daily_heat_stress_day": bool(row.get("heat_stress_day", False)),
            "weather_daily_cold_stress_day": bool(row.get("cold_stress_day", False)),
        })

    start_date = seeding if seeding is not None else weather_df["obs_date"].min()
    history = weather_df[(weather_df["obs_date"] >= start_date) & (weather_df["obs_date"] <= obs_date)]
    if not history.empty:
        features.update({
            "weather_cum_Total_prec_mm": float(history["Total_prec_mm"].sum(skipna=True)),
            "weather_cum_gdd_base10": float(history["gdd_base10"].sum(skipna=True)),
            "weather_cum_rainy_days": int(history["rainy_day"].sum()),
            "weather_cum_heavy_rain_days": int(history["heavy_rain_day"].sum()),
            "weather_cum_heat_stress_days": int(history["heat_stress_day"].sum()),
            "weather_cum_cold_stress_days": int(history["cold_stress_day"].sum()),
        })

    for window_days in (7, 14, 30):
        window_start = obs_date - timedelta(days=window_days - 1)
        window = weather_df[(weather_df["obs_date"] >= window_start) & (weather_df["obs_date"] <= obs_date)]
        if window.empty:
            continue
        features.update({
            f"weather_{window_days}d_Total_prec_mm": float(window["Total_prec_mm"].sum(skipna=True)),
            f"weather_{window_days}d_Temp_mean_C": float(window["Temp_mean_C"].mean(skipna=True)),
            f"weather_{window_days}d_gdd_base10": float(window["gdd_base10"].sum(skipna=True)),
            f"weather_{window_days}d_rainy_days": int(window["rainy_day"].sum()),
        })

    return features


## 5. Extract Sentinel-2 Observation Features

This section keeps one row per field-date observation. It computes broad spectral, vegetation-index, SCL, and quality features that later notebooks can aggregate, sample, or pivot for different crop-mapping models.

In [ ]:
S2_STAT_AGGS = ("mean", "std", "min", "max", "p10", "p25", "p75", "p90", "count")


def safe_divide(num, den, min_abs: float = 1e-10):
    num = np.asarray(num, dtype="float64")
    den = np.asarray(den, dtype="float64")
    out = np.full(np.broadcast_shapes(num.shape, den.shape), np.nan, dtype="float64")
    valid = np.abs(den) > min_abs
    with np.errstate(divide="ignore", invalid="ignore"):
        np.divide(num, den, out=out, where=valid)
    return out


def read_s2_bands(path: Path) -> Optional[dict[str, np.ndarray]]:
    try:
        with rasterio.open(path) as src:
            bands = {}
            for idx, band_name in enumerate(BAND_ORDER, start=1):
                if idx > src.count:
                    break
                arr = src.read(idx).astype("float64")
                if src.nodata is not None:
                    arr[arr == src.nodata] = np.nan
                finite = arr[np.isfinite(arr)]
                if finite.size and np.percentile(finite, 99) > 2.0:
                    arr = arr / 10000.0
                arr[arr <= 0] = np.nan
                bands[band_name] = arr
            return bands if bands else None
    except Exception:
        return None


def find_scl_file(scl_dir: Path, obs_date: date) -> Optional[Path]:
    if not scl_dir.exists():
        return None
    target_name = f"S2_L2A_SCL_{obs_date:%Y%m%d}.tif"
    direct = scl_dir / target_name
    if direct.exists():
        return direct
    for path in scl_dir.glob("S2_L2A_SCL_*.tif"):
        match = SCL_PATTERN.search(path.name)
        if not match:
            continue
        path_date = date(int(match.group(1)), int(match.group(2)), int(match.group(3)))
        if path_date == obs_date:
            return path
    return None


def read_scl(path: Optional[Path], expected_shape: tuple[int, int]) -> Optional[np.ndarray]:
    if path is None:
        return None
    try:
        with rasterio.open(path) as src:
            scl = src.read(1)
        if scl.shape != expected_shape:
            return None
        return scl
    except Exception:
        return None


def s2_stats(values: np.ndarray, prefix: str) -> dict:
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {f"{prefix}_{agg}": np.nan for agg in S2_STAT_AGGS}
    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_std": float(np.std(values)),
        f"{prefix}_min": float(np.min(values)),
        f"{prefix}_max": float(np.max(values)),
        f"{prefix}_p10": float(np.percentile(values, 10)),
        f"{prefix}_p25": float(np.percentile(values, 25)),
        f"{prefix}_p75": float(np.percentile(values, 75)),
        f"{prefix}_p90": float(np.percentile(values, 90)),
        f"{prefix}_count": int(values.size),
    }


def compute_vegetation_indices(bands: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    b01 = bands.get("B01")
    b02 = bands.get("B02")
    b03 = bands.get("B03")
    b04 = bands.get("B04")
    b05 = bands.get("B05")
    b08 = bands.get("B08")
    b11 = bands.get("B11")
    b12 = bands.get("B12")

    indices = {}
    if b08 is not None and b04 is not None:
        indices["NDVI"] = safe_divide(b08 - b04, b08 + b04)
        indices["SAVI"] = safe_divide(1.5 * (b08 - b04), b08 + b04 + 0.5)
    if b08 is not None and b04 is not None and b02 is not None:
        indices["EVI"] = safe_divide(2.5 * (b08 - b04), b08 + 6.0 * b04 - 7.5 * b02 + 1.0)
    if b08 is not None and b03 is not None:
        indices["GNDVI"] = safe_divide(b08 - b03, b08 + b03)
        indices["NDWI_GREEN"] = safe_divide(b03 - b08, b03 + b08)
    if b08 is not None and b05 is not None:
        indices["NDRE"] = safe_divide(b08 - b05, b08 + b05)
    if b08 is not None and b11 is not None:
        indices["NDMI"] = safe_divide(b08 - b11, b08 + b11)
        indices["MSI"] = safe_divide(b11, b08)
    if b08 is not None and b12 is not None:
        indices["NBR"] = safe_divide(b08 - b12, b08 + b12)
    if b11 is not None and b04 is not None and b08 is not None and b02 is not None:
        indices["BSI"] = safe_divide((b11 + b04) - (b08 + b02), (b11 + b04) + (b08 + b02))
    if b01 is not None and b08 is not None:
        indices["CAI_B01_B08"] = safe_divide(b08 - b01, b08 + b01)

    return indices


def scl_features(scl: Optional[np.ndarray], valid_mask: np.ndarray, keep_classes: tuple[int, ...]) -> dict:
    total = int(valid_mask.size)
    output = {
        "s2_total_pixel_count": total,
        "s2_valid_pixel_count": int(valid_mask.sum()),
        "s2_valid_pixel_fraction": float(valid_mask.mean()) if total else np.nan,
    }
    if scl is None:
        for class_id, class_name in SCL_CLASSES.items():
            output[f"scl_{class_name}_fraction"] = np.nan
        output["scl_keep_fraction"] = np.nan
        output["scl_cloud_fraction"] = np.nan
        return output

    for class_id, class_name in SCL_CLASSES.items():
        output[f"scl_{class_name}_fraction"] = float((scl == class_id).mean())
    output["scl_keep_fraction"] = float(np.isin(scl, keep_classes).mean())
    output["scl_cloud_fraction"] = float(np.isin(scl, [7, 8, 9, 10]).mean())
    return output


def extract_s2_observations(
    field_dir: Path,
    meta: dict,
    cfg: CropFeatureStoreConfig,
    weather_df: pd.DataFrame,
) -> list[dict]:
    s2_dir = field_dir / "s2_images"
    scl_dir = field_dir / "scl_masks"
    if not s2_dir.exists():
        return []

    seeding, harvest = metadata_dates(meta)
    rows = []
    s2_files = sorted(path for path in s2_dir.glob("S2_L2A_*.tif") if S2_PATTERN.search(path.name))

    for s2_path in s2_files:
        match = S2_PATTERN.search(s2_path.name)
        obs_date = date(int(match.group(1)), int(match.group(2)), int(match.group(3)))
        position = season_position(obs_date, seeding, harvest)

        # Keep only the declared crop season when dates are available.
        if seeding and harvest and not position["in_declared_season"]:
            continue

        bands = read_s2_bands(s2_path)
        if not bands:
            continue

        ref_band = bands.get("B04")
        if ref_band is None:
            ref_band = next(iter(bands.values()))
        scl = read_scl(find_scl_file(scl_dir, obs_date), ref_band.shape)
        finite_mask = np.isfinite(ref_band)
        if scl is not None:
            valid_mask = finite_mask & np.isin(scl, cfg.scl_keep_classes)
        else:
            valid_mask = finite_mask

        row = {
            "field_id": meta["field_id"],
            "date": obs_date.isoformat(),
            "doy": obs_date.timetuple().tm_yday,
            **position,
            **weather_features_for_date(weather_df, obs_date, seeding),
            **scl_features(scl, valid_mask, cfg.scl_keep_classes),
        }

        for band_name, band in bands.items():
            row.update(s2_stats(band[valid_mask], f"s2_{band_name}"))

        for index_name, index_map in compute_vegetation_indices(bands).items():
            row.update(s2_stats(index_map[valid_mask], f"s2_{index_name}"))

        rows.append(row)

    return rows


## 6. Process Fields in Internal Dask Chunks

Each Dask worker processes a bounded group of fields and writes temporary part files. The notebook combines those parts into the final two Parquet files and deletes the temporary directory, so the final output stays simple.


In [ ]:
import gc


def process_field(metadata_path: Path, cfg: CropFeatureStoreConfig) -> dict:
    start = time.time()
    field_dir = metadata_path.parent
    field_id = metadata_path.stem.replace("metadata-", "")

    try:
        meta = parse_metadata(metadata_path)
        static_features = extract_static_features(field_dir, meta["field_id"])
        weather_df = load_weather_daily(field_dir, meta)
        observation_rows = extract_s2_observations(field_dir, meta, cfg, weather_df)
        field_row = {**meta, **static_features}

        return {
            "success": True,
            "field_id": meta["field_id"],
            "field_row": field_row,
            "observation_rows": observation_rows,
            "error": "",
            "duration_seconds": time.time() - start,
            "n_s2_observations": len(observation_rows),
        }
    except Exception as exc:
        return {
            "success": False,
            "field_id": field_id,
            "field_row": {},
            "observation_rows": [],
            "error": f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}",
            "duration_seconds": time.time() - start,
            "n_s2_observations": 0,
        }


def chunked(paths: list[Path], chunk_size: int) -> list[list[Path]]:
    return [paths[i : i + chunk_size] for i in range(0, len(paths), chunk_size)]


def process_chunk(chunk_index: int, metadata_path_texts: list[str], cfg: CropFeatureStoreConfig, parts_dir: str) -> dict:
    parts_path = Path(parts_dir)
    parts_path.mkdir(parents=True, exist_ok=True)

    field_rows = []
    observation_rows = []
    failures = []
    total_duration = 0.0

    for metadata_path_text in metadata_path_texts:
        result = process_field(Path(metadata_path_text), cfg)
        total_duration += result["duration_seconds"]
        if result["success"]:
            field_rows.append(result["field_row"])
            observation_rows.extend(result["observation_rows"])
        else:
            failures.append({
                "field_id": result["field_id"],
                "error": result["error"][:1000],
            })

    fields_path = parts_path / f"fields_part_{chunk_index:04d}.parquet"
    observations_path = parts_path / f"observations_part_{chunk_index:04d}.parquet"

    fields_df = prepare_for_parquet(pd.DataFrame(field_rows))
    observations_df = prepare_for_parquet(pd.DataFrame(observation_rows))

    fields_df.to_parquet(fields_path, index=False)
    observations_df.to_parquet(observations_path, index=False)

    summary = {
        "chunk_index": chunk_index,
        "fields_path": str(fields_path),
        "observations_path": str(observations_path),
        "requested_fields": len(metadata_path_texts),
        "successful_fields": len(field_rows),
        "failed_fields": len(failures),
        "observation_rows": int(observations_df.shape[0]),
        "field_columns": int(fields_df.shape[1]),
        "observation_columns": int(observations_df.shape[1]),
        "duration_seconds": float(total_duration),
        "first_failure": failures[0] if failures else None,
    }

    del field_rows, observation_rows, fields_df, observations_df
    gc.collect()
    return summary


def combine_part_files(part_paths: list[str], output_path: Path) -> tuple[Path, int, int]:
    existing_paths = [Path(path) for path in part_paths if Path(path).exists()]
    if not existing_paths:
        df = pd.DataFrame()
    else:
        frames = [pd.read_parquet(path) for path in existing_paths]
        df = pd.concat(frames, ignore_index=True, sort=False)
        del frames

    df = prepare_for_parquet(df)
    df.to_parquet(output_path, index=False)
    rows, cols = df.shape
    del df
    gc.collect()
    return output_path, int(rows), int(cols)


def combine_feature_store_parts(summaries: list[dict], output_dir: Path) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    summaries = sorted(summaries, key=lambda item: item["chunk_index"])

    fields_path = output_dir / "crop_mapping_fields.parquet"
    observations_path = output_dir / "crop_mapping_observations.parquet"

    _, field_rows, field_cols = combine_part_files(
        [summary["fields_path"] for summary in summaries],
        fields_path,
    )
    _, observation_rows, observation_cols = combine_part_files(
        [summary["observations_path"] for summary in summaries],
        observations_path,
    )

    print(f"Saved {fields_path.name}: {field_rows} rows x {field_cols} columns")
    print(f"Saved {observations_path.name}: {observation_rows} rows x {observation_cols} columns")

    return {
        "crop_mapping_fields": fields_path,
        "crop_mapping_observations": observations_path,
    }


## 7. Run the Full Crop-Mapping Feature Store Extraction

Run this cell after attaching the YieldSAT dataset on Kaggle. It processes all discovered fields, uses Dask across the available CPU cores, writes temporary internal parts, combines them into the final two Parquet files, and deletes the temporary parts directory.


In [ ]:
import shutil
from dask.distributed import Client, LocalCluster, as_completed

cfg = CropFeatureStoreConfig(scl_keep_classes=SCL_KEEP_CLASSES)

data_root = resolve_data_root(DATA_ROOT)
metadata_paths = discover_fields(data_root)
print("Data root:", data_root)
print("Found metadata files:", len(metadata_paths))

if not metadata_paths:
    raise RuntimeError("No YieldSAT metadata files were found. Check DATA_ROOT and dataset attachment.")

if TEMP_PARTS_DIR.exists():
    shutil.rmtree(TEMP_PARTS_DIR)
TEMP_PARTS_DIR.mkdir(parents=True, exist_ok=True)

field_chunks = chunked(metadata_paths, CHUNK_SIZE)
print("Chunks:", len(field_chunks))
print("Workers:", DASK_N_WORKERS)
print("Threads per worker:", DASK_THREADS_PER_WORKER)

cluster = LocalCluster(
    n_workers=DASK_N_WORKERS,
    threads_per_worker=DASK_THREADS_PER_WORKER,
    processes=True,
    dashboard_address=None,
)
client = Client(cluster)
print("Dask dashboard:", client.dashboard_link or "disabled")

summaries = []
run_start = time.time()
try:
    futures = [
        client.submit(
            process_chunk,
            chunk_index,
            [str(path) for path in paths],
            cfg,
            str(TEMP_PARTS_DIR),
            pure=False,
        )
        for chunk_index, paths in enumerate(field_chunks)
    ]

    for completed_count, future in enumerate(as_completed(futures), start=1):
        summary = future.result()
        summaries.append(summary)

        elapsed = time.time() - run_start
        completed_fields = sum(item["requested_fields"] for item in summaries)
        completed_observations = sum(item["observation_rows"] for item in summaries)
        failed_fields = sum(item["failed_fields"] for item in summaries)
        rate = completed_fields / elapsed if elapsed > 0 else 0
        remaining_fields = len(metadata_paths) - completed_fields
        eta_min = (remaining_fields / rate / 60) if rate > 0 else float("nan")

        print(
            f"Chunk {summary['chunk_index'] + 1}/{len(field_chunks)} complete | "
            f"chunks={completed_count}/{len(field_chunks)} | "
            f"fields={completed_fields}/{len(metadata_paths)} | "
            f"obs={completed_observations} | "
            f"failed={failed_fields} | "
            f"rate={rate:.2f} fields/s | "
            f"ETA={eta_min:.1f} min",
            flush=True,
        )
        if summary["first_failure"]:
            print("First failure in chunk:", summary["first_failure"], flush=True)

finally:
    client.close()
    cluster.close()

if not summaries:
    raise RuntimeError("Dask finished without producing any chunk summaries.")

summaries = sorted(summaries, key=lambda item: item["chunk_index"])
total_failed = sum(item["failed_fields"] for item in summaries)
total_success = sum(item["successful_fields"] for item in summaries)
total_observations = sum(item["observation_rows"] for item in summaries)
elapsed = time.time() - run_start

print(
    f"Extraction complete: {total_success} OK, {total_failed} failed, "
    f"{total_observations} observation rows in {elapsed / 60:.1f} min"
)

output_paths = combine_feature_store_parts(summaries, OUTPUT_DIR)

if TEMP_PARTS_DIR.exists():
    shutil.rmtree(TEMP_PARTS_DIR)
    print("Deleted temporary parts directory:", TEMP_PARTS_DIR)

print("Feature store saved to:", OUTPUT_DIR)


## 8. Verify the Two Final Saved Tables

This cell checks that both expected Parquet tables exist, confirms the temporary parts directory was removed, and prints row/column counts plus crop label distribution.


In [ ]:
required_outputs = {
    "crop_mapping_fields": "crop_mapping_fields.parquet",
    "crop_mapping_observations": "crop_mapping_observations.parquet",
}

for name, filename in required_outputs.items():
    path = output_paths[name]
    if path.name != filename:
        raise RuntimeError(f"Unexpected output path for {name}: {path}")
    if not path.exists():
        raise FileNotFoundError(f"Missing expected output: {path}")
    df = pd.read_parquet(path)
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")

if TEMP_PARTS_DIR.exists():
    raise RuntimeError(f"Temporary parts directory still exists: {TEMP_PARTS_DIR}")

fields_df = pd.read_parquet(output_paths["crop_mapping_fields"])
print("\nCrop counts:")
print(fields_df["crop"].value_counts(dropna=False))

print("\nFinal output files:")
for name in required_outputs.values():
    path = OUTPUT_DIR / name
    print("-", path.name, f"({path.stat().st_size / 1024 / 1024:.2f} MB)")


## 9. Optional Zip for Download

This optional cell creates one zip containing the two final Parquet outputs. `SAVE_ZIP` is disabled by default so the normal notebook output remains only the two Parquet files.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

if SAVE_ZIP:
    zip_path = OUTPUT_DIR.parent / "yieldsat_crop_mapping_feature_store.zip"
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
        for path in output_paths.values():
            zf.write(path, arcname=path.name)

    print("Created:", zip_path)
    print("Zip size MB:", zip_path.stat().st_size / 1024 / 1024)
    print("Contents:")
    with ZipFile(zip_path) as zf:
        for name in zf.namelist():
            print("-", name)
else:
    print("SAVE_ZIP is False; skipping zip export.")